# S4 — True Null Calibration Validation

The S3 permutation test has only 32 True Null cases (f=0, constant noise),
yielding an observed FP rate of 12.5% (4/32) with a 95% CI of [5%, 28%].
This is too few to verify whether the method is properly calibrated at 5%.

This notebook generates **500 additional True Null cases** per x_distribution
(4,000 total), runs the same 4-metric permutation test, and checks FP calibration.

**True Null definition**: y = N(0,1) independent of x. No mean signal, no heteroscedasticity.

| Parameter | Value |
|-----------|-------|
| f(x) | 0 (constant) |
| spread_pattern | constant |
| noise | N(0,1) |
| n_points | 500 |
| n_permutations | 500 |
| x_distributions | 8 types × 500 cases each |

In [23]:
from __future__ import annotations

import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from scipy.stats import rankdata

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it, **kw):
        return it

warnings.filterwarnings('ignore')

DATA_DIR = Path('generated_scatterplot_data')
OUT_DIR  = DATA_DIR / 'full' / 'S4_null_validate'
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_POINTS    = 500
N_PERM      = 500
N_PER_XDIST = 500
SEED_BASE   = 99_000_000

X_DISTRIBUTIONS = ['even', 'left_dense', 'right_dense', 'center_dense',
                   'clusters_2', 'clusters_3', 'clusters_4', 'clusters_5']

print(f'Will generate {len(X_DISTRIBUTIONS)} × {N_PER_XDIST} = {len(X_DISTRIBUTIONS)*N_PER_XDIST} True Null cases')
print(f'Each: {N_POINTS} points, {N_PERM} permutations, 4 metrics')

Will generate 8 × 500 = 4000 True Null cases
Each: 500 points, 500 permutations, 4 metrics


## 1. Generate True Null Data

Replicate the same x-sampling as S1 but with y = N(0,1) independent of x.

In [24]:
CLUSTER_OCCUPANCY = 0.4
CLUSTER_BACKGROUND_FRAC = 0.1


def _cluster_bounds(n_clusters):
    cluster_width = CLUSTER_OCCUPANCY / n_clusters
    gap_width = (1.0 - CLUSTER_OCCUPANCY) / (n_clusters + 1)
    bounds = []
    for i in range(n_clusters):
        lo = gap_width * (i + 1) + cluster_width * i
        hi = lo + cluster_width
        bounds.append((lo, hi))
    return bounds


def sample_x(n, distribution, rng):
    if distribution == 'even':
        return rng.uniform(0.0, 1.0, size=n)
    if distribution == 'left_dense':
        return rng.beta(2.0, 5.0, size=n)
    if distribution == 'right_dense':
        return rng.beta(5.0, 2.0, size=n)
    if distribution == 'center_dense':
        return rng.beta(5.0, 5.0, size=n)
    if distribution.startswith('clusters_'):
        n_clusters = int(distribution.split('_')[1])
        bounds = _cluster_bounds(n_clusters)
        is_bg = rng.random(size=n) < CLUSTER_BACKGROUND_FRAC
        n_bg = is_bg.sum()
        component = rng.integers(0, n_clusters, size=n)
        x = np.empty(n, dtype=float)
        for i, (lo, hi) in enumerate(bounds):
            mask = (~is_bg) & (component == i)
            x[mask] = rng.uniform(lo, hi, size=mask.sum())
        x[is_bg] = rng.uniform(0.0, 1.0, size=n_bg)
        return x
    raise ValueError(f'Unknown x distribution: {distribution}')


print('Samplers ready')

Samplers ready


In [25]:
# Generate all True Null (x, y) pairs
n_total = len(X_DISTRIBUTIONS) * N_PER_XDIST
x_all = np.empty((n_total, N_POINTS), dtype=np.float64)
y_all = np.empty((n_total, N_POINTS), dtype=np.float64)
meta_rows = []

idx = 0
for xd in X_DISTRIBUTIONS:
    for k in range(N_PER_XDIST):
        seed = SEED_BASE + idx
        rng = np.random.default_rng(seed)
        x = sample_x(N_POINTS, xd, rng)
        y = rng.normal(0.0, 1.0, size=N_POINTS)  # True Null: y independent of x
        x_all[idx] = x
        y_all[idx] = y
        meta_rows.append({'null_id': idx, 'x_distribution': xd, 'seed': seed})
        idx += 1

meta_df = pd.DataFrame(meta_rows)
print(f'Generated {n_total} True Null cases')
print(meta_df['x_distribution'].value_counts().sort_index().to_string())

Generated 4000 True Null cases
x_distribution
center_dense    500
clusters_2      500
clusters_3      500
clusters_4      500
clusters_5      500
even            500
left_dense      500
right_dense     500


## 2. Permutation Test (same procedure as S3)

Phase 1: |Pearson|, |Spearman|, η² (vectorised)  
Phase 2: dcor (loop)  
Phase 3: Z-scores + joint test

In [26]:
def _double_center(a):
    D = squareform(pdist(a.reshape(-1, 1)))
    return D - D.mean(axis=0, keepdims=True) - D.mean(axis=1, keepdims=True) + D.mean()


def _precompute_bins(x, n_bins=10, min_count=5):
    try:
        bins = np.asarray(
            pd.cut(x, bins=n_bins, labels=False, include_lowest=True, duplicates='drop'),
            dtype=float,
        )
    except Exception:
        return None
    valid_ids = np.unique(bins[~np.isnan(bins)]).astype(int)
    masks, counts = [], []
    for b in valid_ids:
        m = bins == b
        if m.sum() >= min_count:
            masks.append(m)
            counts.append(m.sum())
    if len(masks) < 2:
        return None
    B_ind = np.array([m.astype(np.float64) for m in masks])
    return B_ind, np.array(counts, dtype=np.float64)


def _generate_perms(n, n_perm, seed):
    rng = np.random.default_rng(seed)
    return np.array([rng.permutation(n) for _ in range(n_perm)])


print('Helpers ready')

Helpers ready


In [27]:
# Phase 1: |Pearson|, |Spearman|, η²
pearson_obs  = np.empty(n_total)
spearman_obs = np.empty(n_total)
eta2_obs     = np.empty(n_total)
pearson_null  = np.empty((n_total, N_PERM), dtype=np.float32)
spearman_null = np.empty((n_total, N_PERM), dtype=np.float32)
eta2_null     = np.empty((n_total, N_PERM), dtype=np.float32)

t0 = time.time()
for i in tqdm(range(n_total), desc='Phase 1'):
    x = x_all[i]
    y = y_all[i]
    perms = _generate_perms(len(x), N_PERM, SEED_BASE + n_total + i)

    xc = x - x.mean(); yc = y - y.mean()
    sx = np.sqrt((xc**2).sum()); sy = np.sqrt((yc**2).sum())
    if sx > 0 and sy > 0:
        d = sx * sy
        pearson_obs[i]  = abs(float((xc*yc).sum() / d))
        pearson_null[i] = np.abs((xc * yc[perms]).sum(axis=1) / d)
    else:
        pearson_obs[i] = 0.0; pearson_null[i] = 0.0

    xr = rankdata(x).astype(np.float64); yr = rankdata(y).astype(np.float64)
    xrc = xr - xr.mean(); yrc = yr - yr.mean()
    sxr = np.sqrt((xrc**2).sum()); syr = np.sqrt((yrc**2).sum())
    if sxr > 0 and syr > 0:
        dr = sxr * syr
        spearman_obs[i]  = abs(float((xrc*yrc).sum() / dr))
        spearman_null[i] = np.abs((xrc * yrc[perms]).sum(axis=1) / dr)
    else:
        spearman_obs[i] = 0.0; spearman_null[i] = 0.0

    bin_info = _precompute_bins(x)
    y_mean = float(y.mean()); ss_total = float(((y - y_mean)**2).sum())
    if bin_info is not None and ss_total > 0:
        B_ind, bin_counts = bin_info
        bm_obs = (B_ind @ y) / bin_counts
        eta2_obs[i] = float((bin_counts * (bm_obs - y_mean)**2).sum() / ss_total)
        y_perms = y[perms]
        bm_null = (y_perms @ B_ind.T) / bin_counts
        ss_bet = (bin_counts * (bm_null - y_mean)**2).sum(axis=1)
        eta2_null[i] = (ss_bet / ss_total).astype(np.float32)
    else:
        eta2_obs[i] = 0.0; eta2_null[i] = 0.0

print(f'Phase 1 done in {(time.time()-t0)/60:.1f} min')

Phase 1: 100%|██████████| 4000/4000 [00:24<00:00, 164.50it/s]

Phase 1 done in 0.4 min


In [28]:
# Phase 2: dcor
dcor_obs  = np.empty(n_total)
dcor_null = np.empty((n_total, N_PERM), dtype=np.float32)

t0 = time.time()
for i in tqdm(range(n_total), desc='Phase 2'):
    x = x_all[i]; y = y_all[i]
    perms = _generate_perms(len(x), N_PERM, SEED_BASE + n_total + i)

    A = _double_center(x); B = _double_center(y)
    dcov_xx = np.sqrt(max((A*A).mean(), 0))
    dcov_yy = np.sqrt(max((B*B).mean(), 0))
    dcov_denom = np.sqrt(dcov_xx * dcov_yy)

    dcov_obs = np.sqrt(max(float((A*B).mean()), 0))
    dcor_obs[i] = float(dcov_obs / dcov_denom) if dcov_denom > 0 else 0.0

    if dcov_denom > 0:
        for k in range(N_PERM):
            p = perms[k]
            dcov2 = float((A * B[p][:, p]).mean())
            dcor_null[i, k] = np.sqrt(max(dcov2, 0)) / dcov_denom
    else:
        dcor_null[i] = 0.0

    if (i+1) % 500 == 0:
        el = time.time() - t0
        rate = (i+1) / el
        eta_h = (n_total - i - 1) / rate / 3600
        print(f'  {i+1:>5,}/{n_total:,}  ({rate:.1f} cases/s, ETA {eta_h:.1f} h)')

print(f'Phase 2 done in {(time.time()-t0)/3600:.1f} hours')

Phase 2:  12%|█▎        | 500/4000 [03:04<21:51,  2.67it/s]

    500/4,000  (2.7 cases/s, ETA 0.4 h)


Phase 2:  25%|██▌       | 1000/4000 [06:04<17:18,  2.89it/s]

  1,000/4,000  (2.7 cases/s, ETA 0.3 h)


Phase 2:  38%|███▊      | 1500/4000 [09:10<14:43,  2.83it/s]

...

  3,000/4,000  (2.7 cases/s, ETA 0.1 h)


Phase 2:  88%|████████▊ | 3500/4000 [21:05<03:25,  2.43it/s]

  3,500/4,000  (2.8 cases/s, ETA 0.1 h)


Phase 2: 100%|██████████| 4000/4000 [24:18<00:00,  2.74it/s]

  4,000/4,000  (2.7 cases/s, ETA 0.0 h)
Phase 2 done in 0.4 hours


In [29]:
# Phase 3: Z-scores + joint test
names = ['pearson', 'spearman', 'dcor', 'eta2']
all_obs  = [pearson_obs, spearman_obs, dcor_obs, eta2_obs]
all_null = [pearson_null, spearman_null, dcor_null, eta2_null]

results = []
for i in tqdm(range(n_total), desc='Phase 3'):
    row = {'null_id': i, 'x_distribution': meta_df.iloc[i]['x_distribution']}
    z_obs_list, z_null_list = [], []

    for name, obs_arr, null_arr in zip(names, all_obs, all_null):
        obs  = float(obs_arr[i])
        null = null_arr[i].astype(np.float64)
        med = float(np.median(null))
        iqr = float(np.percentile(null, 75) - np.percentile(null, 25))
        if iqr < 1e-12:
            iqr = float(np.std(null)) * 1.35
        if iqr < 1e-12:
            z_o, z_n = 0.0, np.zeros(N_PERM)
        else:
            z_o = (obs - med) / iqr
            z_n = (null - med) / iqr
        z_obs_list.append(z_o)
        z_null_list.append(z_n)
        row[f'{name}_obs'] = obs
        row[f'z_{name}'] = float(z_o)

    T_obs  = max(z_obs_list)
    T_null = np.stack(z_null_list).max(axis=0)
    p_value = float(np.sum(T_null >= T_obs) + 1) / (N_PERM + 1)

    row['T_joint'] = float(T_obs)
    row['p_value'] = float(p_value)
    row['classification'] = 'detectable' if p_value <= 0.05 else ('not_detectable' if p_value >= 0.10 else 'uncertain')

    # Which metric drives T_joint?
    driver_idx = int(np.argmax(z_obs_list))
    row['driver'] = names[driver_idx]
    results.append(row)

result_df = pd.DataFrame(results)
result_df.to_parquet(OUT_DIR / 'null_validate.parquet', index=False)
print(f'Saved {len(result_df)} results to {OUT_DIR}/null_validate.parquet')

Phase 3:   0%|          | 0/4000 [00:00<?, ?it/s]

Phase 3: 100%|██████████| 4000/4000 [00:01<00:00, 2380.80it/s]

Saved 4000 results to generated_scatterplot_data/full/S4_null_validate/null_validate.parquet


In [30]:
# Save raw null arrays for S5 visualization analysis
np.savez_compressed(
    OUT_DIR / '_null_phase1.npz',
    pearson_obs=pearson_obs, spearman_obs=spearman_obs, eta2_obs=eta2_obs,
    pearson_null=pearson_null, spearman_null=spearman_null, eta2_null=eta2_null,
)
np.savez_compressed(
    OUT_DIR / '_null_phase2.npz',
    dcor_obs=dcor_obs, dcor_null=dcor_null,
)
meta_df.to_csv(OUT_DIR / 'null_meta.csv', index=False)
print(f'Saved NPZ checkpoints and metadata to {OUT_DIR}/')

Saved NPZ checkpoints and metadata to generated_scatterplot_data/full/S4_null_validate/


## 3. FP Rate Analysis

Merge S4's 4,000 new True Null results with S3's original 32 True Null cases for a combined calibration check.

In [31]:
# --- Load S3's original 32 True Null results ---
s3_df = pd.read_parquet(DATA_DIR / 'full' / 'S3' / 'permutation_test.parquet')
cases_df = pd.read_csv(DATA_DIR / 'cases.csv')

# True Null in S3: family_id == 'Null' AND spread_pattern == 'constant'
null_mask = (cases_df['family_id'] == 'Null') & (cases_df['spread_pattern'] == 'constant')
s3_true_null = s3_df[null_mask.values].copy()
s3_true_null['source'] = 'S3'
print(f'S3 True Null cases: {len(s3_true_null)}')
print(f'  FP: {(s3_true_null["classification"] == "detectable").sum()}/{len(s3_true_null)}')

# --- S4 results ---
result_df['source'] = 'S4'
print(f'\nS4 True Null cases: {len(result_df)}')
print(f'  FP: {(result_df["classification"] == "detectable").sum()}/{len(result_df)}')

# --- Merge ---
# S3 doesn't have x_distribution directly, get it from cases_df
s3_true_null = s3_true_null.copy()
s3_true_null['x_distribution'] = cases_df.loc[null_mask.values, 'x_distribution'].values

# Add driver column to S3
s3_z_cols = ['z_pearson', 'z_spearman', 'z_dcor', 'z_eta2']
s3_true_null['driver'] = s3_true_null[s3_z_cols].idxmax(axis=1).str.replace('z_', '')

combined_cols = ['x_distribution', 'p_value', 'classification', 'T_joint',
                 'z_pearson', 'z_spearman', 'z_dcor', 'z_eta2', 'source', 'driver']
combined = pd.concat([
    s3_true_null[combined_cols],
    result_df[combined_cols]
], ignore_index=True)

n_combined = len(combined)
n_fp_combined = (combined['classification'] == 'detectable').sum()
fp_rate_combined = n_fp_combined / n_combined

# Wilson CI
z_val = 1.96
p_hat = fp_rate_combined
denom = 1 + z_val**2 / n_combined
center = (p_hat + z_val**2 / (2*n_combined)) / denom
margin = z_val * np.sqrt((p_hat*(1-p_hat) + z_val**2/(4*n_combined)) / n_combined) / denom
ci_lo, ci_hi = center - margin, center + margin

print('\n' + '='*60)
print(f'COMBINED TRUE NULL FP CALIBRATION (S3 + S4, n={n_combined})')
print('='*60)
print(f'FP (p≤0.05): {n_fp_combined}/{n_combined} = {fp_rate_combined:.2%}')
print(f'95% CI (Wilson): [{ci_lo:.2%}, {ci_hi:.2%}]')
print(f'Target: 5.0%')
print()
print('Classification distribution:')
print(combined['classification'].value_counts().to_string())
print()
print('Breakdown by source:')
for src in ['S3', 'S4']:
    sub = combined[combined['source'] == src]
    fp = (sub['classification'] == 'detectable').sum()
    print(f'  {src}: {fp}/{len(sub)} = {fp/len(sub):.1%}')

S3 True Null cases: 32
  FP: 4/32

S4 True Null cases: 4000
  FP: 213/4000

COMBINED TRUE NULL FP CALIBRATION (S3 + S4, n=4032)
FP (p≤0.05): 217/4032 = 5.38%
95% CI (Wilson): [4.73%, 6.12%]
Target: 5.0%

Classification distribution:
classification
not_detectable    3609
detectable         217
uncertain          206

Breakdown by source:
  S3: 4/32 = 12.5%
  S4: 213/4000 = 5.3%


In [32]:
# FP rate by x_distribution (combined S3 + S4)
print('FP rate by x_distribution (combined S3 + S4):')
print(f'{"x_distribution":>15s}  {"n":>5s}  {"FP":>4s}  {"rate":>7s}  {"CI_lo":>6s}  {"CI_hi":>6s}')
print('-'*55)

for xd in X_DISTRIBUTIONS:
    sub = combined[combined['x_distribution'] == xd]
    n_sub = len(sub)
    fp_sub = (sub['classification'] == 'detectable').sum()
    rate = fp_sub / n_sub
    p_h = rate
    d = 1 + z_val**2/n_sub
    c = (p_h + z_val**2/(2*n_sub)) / d
    m = z_val * np.sqrt((p_h*(1-p_h) + z_val**2/(4*n_sub))/n_sub) / d
    print(f'{xd:>15s}  {n_sub:>5d}  {fp_sub:>4d}  {rate:>6.1%}  {c-m:>5.1%}  {c+m:>5.1%}')

FP rate by x_distribution (combined S3 + S4):
 x_distribution      n    FP     rate   CI_lo   CI_hi
-------------------------------------------------------
           even    504    29    5.8%   4.0%   8.1%
     left_dense    504    21    4.2%   2.7%   6.3%
    right_dense    504    33    6.5%   4.7%   9.1%
   center_dense    504    22    4.4%   2.9%   6.5%
     clusters_2    504    30    6.0%   4.2%   8.4%
     clusters_3    504    22    4.4%   2.9%   6.5%
     clusters_4    504    25    5.0%   3.4%   7.2%
     clusters_5    504    35    6.9%   5.0%   9.5%


In [33]:
# Among FP cases: which metric drives the false detection? (combined)
fp_cases = combined[combined['classification'] == 'detectable']
print(f'FP driver metric distribution (combined, n={len(fp_cases)}):')
if len(fp_cases) > 0:
    print(fp_cases['driver'].value_counts().to_string())
    print()
    print('FP driver by x_distribution:')
    ct = pd.crosstab(fp_cases['x_distribution'], fp_cases['driver'])
    print(ct.to_string())
else:
    print('  No false positives!')

FP driver metric distribution (combined, n=217):
driver
dcor        120
eta2         67
pearson      20
spearman     10

FP driver by x_distribution:
driver          dcor  eta2  pearson  spearman
x_distribution                               
center_dense      12     8        2         0
clusters_2        18     8        2         2
clusters_3         7     7        5         3
clusters_4        16     7        0         2
clusters_5        18    12        5         0
even              17     8        2         2
left_dense        11     8        2         0
right_dense       21     9        2         1


In [ ]:
# p-value distribution under True Null — should be uniform (combined)
import matplotlib.pyplot as plt
%matplotlib inline

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# 1. p-value histogram (should be uniform)
ax = axes[0]
ax.hist(combined['p_value'], bins=50, color='steelblue', edgecolor='white', density=True)
ax.axhline(1.0, color='red', ls='--', lw=1.5, label='Uniform(0,1)')
ax.set_xlabel('p-value')
ax.set_ylabel('Density')
ax.set_title(f'p-value Distribution Under True Null (n={n_combined})')
ax.legend()

# 2. p-value CDF vs uniform
ax = axes[1]
p_sorted = np.sort(combined['p_value'].values)
ecdf = np.arange(1, len(p_sorted)+1) / len(p_sorted)
ax.plot(p_sorted, ecdf, 'b-', lw=1.5, label='Empirical CDF')
ax.plot([0, 1], [0, 1], 'r--', lw=1, label='Uniform')
ax.set_xlabel('p-value')
ax.set_ylabel('Cumulative fraction')
ax.set_title('p-value CDF vs Uniform')
ax.legend()

# 3. FP rate by x_distribution (combined)
ax = axes[2]
fp_rates = []
for xd in X_DISTRIBUTIONS:
    sub = combined[combined['x_distribution'] == xd]
    fp_rates.append((sub['classification'] == 'detectable').mean())
ax.barh(range(len(X_DISTRIBUTIONS)), fp_rates, color='steelblue', alpha=0.7)
ax.axvline(0.05, color='red', ls='--', lw=1.5, label='α=0.05')
ax.set_yticks(range(len(X_DISTRIBUTIONS)))
ax.set_yticklabels(X_DISTRIBUTIONS)
ax.set_xlabel('FP rate')
ax.set_title('FP Rate by x_distribution (S3+S4)')
ax.legend()

plt.tight_layout()
fig.savefig(OUT_DIR / 'null_calibration.png', bbox_inches='tight')
plt.show()
print(f'Saved → {OUT_DIR}/null_calibration.png')

In [35]:
# Per-metric individual FP rate (not joint, each metric alone)
# Uses S4's raw null arrays (S3's raw arrays are in separate NPZ files)
print(f'Per-metric individual FP rate (S4 only, n={n_total}):')
print()

for mname, obs_arr, null_arr in [('|Pearson|', pearson_obs, pearson_null),
                                  ('|Spearman|', spearman_obs, spearman_null),
                                  ('dcor', dcor_obs, dcor_null),
                                  ('η²', eta2_obs, eta2_null)]:
    single_p = np.array([
        (np.sum(null_arr[i].astype(np.float64) >= obs_arr[i]) + 1) / (N_PERM + 1)
        for i in range(n_total)
    ])
    fp = (single_p <= 0.05).mean()
    p_h = fp; d = 1 + z_val**2 / n_total
    c = (p_h + z_val**2 / (2*n_total)) / d
    m = z_val * np.sqrt((p_h*(1-p_h) + z_val**2/(4*n_total)) / n_total) / d
    print(f'  {mname:12s}  FP={fp:.2%}  CI=[{c-m:.2%}, {c+m:.2%}]  (target: ~5%)')

Per-metric individual FP rate (S4 only, n=4000):

  |Pearson|     FP=5.15%  CI=[4.51%, 5.88%]  (target: ~5%)
  |Spearman|    FP=4.98%  CI=[4.34%, 5.69%]  (target: ~5%)
  dcor          FP=5.33%  CI=[4.67%, 6.06%]  (target: ~5%)
  η²            FP=5.17%  CI=[4.53%, 5.91%]  (target: ~5%)


## 4. Conclusion

In [36]:
print('='*60)
print('CALIBRATION SUMMARY (Combined S3 + S4)')
print('='*60)
print()
print(f'Total True Null cases: {n_combined} (S3: {len(s3_true_null)}, S4: {len(result_df)})')
print(f'Joint test FP rate: {fp_rate_combined:.2%} (target: 5.0%)')
print(f'95% CI: [{ci_lo:.2%}, {ci_hi:.2%}]')
print()
if ci_lo <= 0.05 <= ci_hi:
    print('→ 5% is WITHIN the confidence interval. Method appears calibrated.')
elif fp_rate_combined > 0.05:
    print(f'→ 5% is BELOW the CI. Method may be slightly anti-conservative (FP > 5%).')
    print(f'  Consider: this could be driven by specific x_distributions.')
else:
    print(f'→ 5% is ABOVE the CI. Method is conservative (FP < 5%).')

CALIBRATION SUMMARY (Combined S3 + S4)

Total True Null cases: 4032 (S3: 32, S4: 4000)
Joint test FP rate: 5.38% (target: 5.0%)
95% CI: [4.73%, 6.12%]

→ 5% is WITHIN the confidence interval. Method appears calibrated.
